In [32]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import random

In [35]:
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; ARM Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 Edg/131.0.2903.86",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36 Edg/130.0.2849.68",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 Edg/131.0.2903.86",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.2739.79",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 Edg/131.0.2903.86",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:133.0) Gecko/20100101 Firefox/133.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:133.0) Gecko/20100101 Firefox/133.0",
    "Mozilla/5.0 (X11; Linux x86_64; rv:133.0) Gecko/20100101 Firefox/133.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.1 Safari/605.1.15",
    "Mozilla/5.0 (Macintosh; PPC Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.5 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 OPR/116.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 Brave/1.73.91"
]

In [37]:
url = "https://www.basketball-reference.com/teams/"

headers = {'User-Agent': random.choice(user_agents)}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table', id='teams_active')

main_rows = table.find_all('tr', class_='full_table')

team_links = []

for row in main_rows:
    a_tag = row.find('a')
    href = a_tag.get('href')

    full_url = f"https://www.basketball-reference.com{href}"
        
    team_links.append(full_url)

team_links

['https://www.basketball-reference.com/teams/ATL/',
 'https://www.basketball-reference.com/teams/BOS/',
 'https://www.basketball-reference.com/teams/NJN/',
 'https://www.basketball-reference.com/teams/CHA/',
 'https://www.basketball-reference.com/teams/CHI/',
 'https://www.basketball-reference.com/teams/CLE/',
 'https://www.basketball-reference.com/teams/DAL/',
 'https://www.basketball-reference.com/teams/DEN/',
 'https://www.basketball-reference.com/teams/DET/',
 'https://www.basketball-reference.com/teams/GSW/',
 'https://www.basketball-reference.com/teams/HOU/',
 'https://www.basketball-reference.com/teams/IND/',
 'https://www.basketball-reference.com/teams/LAC/',
 'https://www.basketball-reference.com/teams/LAL/',
 'https://www.basketball-reference.com/teams/MEM/',
 'https://www.basketball-reference.com/teams/MIA/',
 'https://www.basketball-reference.com/teams/MIL/',
 'https://www.basketball-reference.com/teams/MIN/',
 'https://www.basketball-reference.com/teams/NOH/',
 'https://ww

In [ ]:
headers = {'User-Agent': random.choice(user_agents)}

all_teams_data = []

for url in team_links:
    print(url)

    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    team_info = {}
    team_info['Team'] = soup.find('h1').text.strip()

    for p in soup.find_all('p'):
        label = p.find('strong')
        if label and label.text.strip().endswith(':'):
            field_name = label.text.strip()[:-1]
            if field_name == 'Site Last Updated':
                continue
            full_text = p.get_text(separator=' ', strip=True)
            field_value = full_text.replace(label.text.strip(), '', 1).strip()
            team_info[field_name] = ' '.join(field_value.split())

    all_teams_data.append(team_info)

    time.sleep(2)

df = pd.DataFrame(all_teams_data)

df.to_csv('teams_basic_info.csv', index=False)

print("\nDone. Data saved to teams_basic_info.csv")

https://www.basketball-reference.com/teams/ATL/
https://www.basketball-reference.com/teams/BOS/

Done. Data saved to teams_basic_info.csv


In [ ]:
url = "https://www.basketball-reference.com/contracts/"

headers = {'User-Agent': random.choice(user_agents)}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table', id='team_summary')


header_row = table.find('thead').find_all('tr')[-1]
columns = [th.text.strip() for th in header_row.find_all('th')]

data = []
rows = table.find('tbody').find_all('tr')

for row in rows:
    cells = row.find_all(['th', 'td'])

    row_data = [cell.text.strip() for cell in cells]
    data.append(row_data)

df = pd.DataFrame(data, columns=columns)

df.to_csv('team_contracts.csv', index=False)

print("Done. Data saved to team_contracts.csv")

Done. Data saved to team_contracts.csv


In [ ]:
url = "https://www.basketball-reference.com/leagues/"

headers = {'User-Agent': random.choice(user_agents)}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

table = soup.find("table", id="stats")

thead_rows = table.find_all("tr", class_="thead")
header_row = thead_rows[-1]
columns = [th.text.strip() for th in header_row.find_all("th")]

data = []
rows = table.find_all("tr")

for row in rows:
    if "thead" in row.get("class", []) or "over_header" in row.get("class", []):
        continue

    cells = row.find_all(["th", "td"])

    row_data = [cell.text.strip() for cell in cells]

    data.append(row_data)

df = pd.DataFrame(data, columns=columns)

df.to_csv('league.csv', index=False)

print("Done. Data saved to league.csv")

Done. Data saved to league.csv


In [17]:
teams = [
    "ATL", "BOS", "NJN", "CHA", "CHI",
    "CLE", "DAL", "DEN", "DET", "GSW",
    "HOU", "IND", "LAC", "LAL", "MEM",
    "MIA", "MIL", "MIN", "NOH", "NYK",
    "OKC", "ORL", "PHI", "PHO", "POR",
    "SAC", "SAS", "TOR", "UTA", "WAS"
]

years = [2026, 2025, 2024, 2023]

all_links = []
for year in years:
    for team in teams:
        link = f"https://www.basketball-reference.com/teams/{team}/{year}.html"
        all_links.append(link)

for link in all_links:
    print(link)

https://www.basketball-reference.com/teams/ATL/2026.html
https://www.basketball-reference.com/teams/BOS/2026.html
https://www.basketball-reference.com/teams/NJN/2026.html
https://www.basketball-reference.com/teams/CHA/2026.html
https://www.basketball-reference.com/teams/CHI/2026.html
https://www.basketball-reference.com/teams/CLE/2026.html
https://www.basketball-reference.com/teams/DAL/2026.html
https://www.basketball-reference.com/teams/DEN/2026.html
https://www.basketball-reference.com/teams/DET/2026.html
https://www.basketball-reference.com/teams/GSW/2026.html
https://www.basketball-reference.com/teams/HOU/2026.html
https://www.basketball-reference.com/teams/IND/2026.html
https://www.basketball-reference.com/teams/LAC/2026.html
https://www.basketball-reference.com/teams/LAL/2026.html
https://www.basketball-reference.com/teams/MEM/2026.html
https://www.basketball-reference.com/teams/MIA/2026.html
https://www.basketball-reference.com/teams/MIL/2026.html
https://www.basketball-referenc

In [ ]:
from io import StringIO

table_ids = ['roster', 'team_and_opponent', 'team_misc', 'per_game_stats', 'totals_stats',
'per_minute_stats', 'per_poss', 'advanced', 'adj_shooting', 'shooting', 'pbp_stats']

headers = {'User-Agent': random.choice(user_agents)}

years = sorted({re.search(r'/(\d{4})\.html', url).group(1) for url in all_links})

for year in years:
    tables = []

    for url in all_links:
        if not url.endswith(f'/{year}.html'):
            continue

        team = re.search(r'/teams/([A-Z]{3})/', url).group(1)
        print(f'{team} {year}')

        response = requests.get(url, headers=headers)

        response = response.text.replace('<!--', '').replace('-->', '')
        soup = BeautifulSoup(response, 'html.parser')

        for table_id in table_ids:
            table = soup.find('table', id=table_id)

            df = pd.read_html(StringIO(str(table)), flavor='lxml')[0]

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = ['_'.join(c for c in col if 'Unnamed' not in c) for col in df.columns]

            df['Year'] = year
            df['Team'] = team
            df['Table_Type'] = table_id
            tables.append(df)

        time.sleep(3)

    if tables:
        pd.concat(tables).to_csv(f'{year}_All_Teams.csv', index=False)
        print(f'Saved {year}')

ATL 2023
BOS 2023


KeyboardInterrupt: 

In [1]:
for url in team_links:
    print(url)
    response = requests.get(url, headers=headers, timeout=10)
    html_content = response.text.replace('<!--', '').replace('-->', '')
    soup = BeautifulSoup(html_content, 'lxml')

    target = None
    for table in soup.find_all('table'):
        thead = table.find('thead')
        if thead and 'Season' in thead.get_text():
            target = table
            break

    if target is None:
        print(f"No season table found for {url}")
        time.sleep(3)
        continue

    df = pd.read_html(str(target), flavor='lxml')[0]
    df['Team_Abbr'] = url.strip('/').split('/')[-1]
    all_data.append(df)

    time.sleep(3)

final = pd.concat(all_data)
final.to_csv('All_Teams_Main.csv', index=False)

NameError: name 'team_links' is not defined

In [1]:
url = 'https://www.basketball-reference.com/teams/WAS/2023.html'

headers = {'User-Agent': random.choice(user_agents)}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

NameError: name 'random' is not defined